# Silver / Gold layer write helpers
##
# What we do here:
##   - Remove technical columns we don't want to keep (ingest time, source etc.)
##   - Full overwrite: replace entire table with new data + allow schema changes
##   - Incremental merge: upsert records using business key
##     → update if key exists, insert if new
##     → create table if it doesn't exist yet
##
# Why:
##   - Keep clean final tables without extra metadata
##   - Support both full refresh and daily delta loads
##   - Safe & efficient updates without duplicates

In [0]:

from delta.tables import DeltaTable


# Drop technical metadata columns
def drop_metadata_columns(df, metadata_cols):
    return df.drop(*metadata_cols)

# Full load write (overwrite)
def write_full_overwrite(df, target_table):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )
    
# Incremental write using MERGE (upsert logic)
def write_incremental_merge(
    df,
    target_table,
    business_key
):
    spark = df.sparkSession
    # If table does not exist → create it
    if not spark.catalog.tableExists(target_table):
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(target_table)
        )
        return
    delta_table = DeltaTable.forName(spark, target_table)
    (
        delta_table.alias("target")
        .merge(
            df.alias("source"),
            f"target.{business_key} = source.{business_key}"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )